In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def evaluate_clustering(X_data, labels, y_true, model_name, ignore_noise=False):
    """Computes Internal (Silhouette) and External (Purity & Entropy) indices."""
    if ignore_noise:
        mask = labels != -1
        X_eval, labels_eval, y_eval = X_data[mask], labels[mask], y_true[mask]
    else:
        X_eval, labels_eval, y_eval = X_data, labels, y_true

    unique_clusters = set(labels_eval)
    if len(unique_clusters) > 1 and len(unique_clusters) < len(X_eval):
        sil = silhouette_score(X_eval, labels_eval)
    else:
        sil = float("nan")

    # Contingency matrix (rows = predicted clusters, cols = true species)
    ct = pd.crosstab(labels_eval, y_eval).values
    n_total = ct.sum()

    # Purity: sum of max class count in each cluster / total samples
    purity = np.sum(np.max(ct, axis=1)) / n_total

    # Entropy: weighted sum of cluster entropies
    cluster_sizes = ct.sum(axis=1, keepdims=True)
    probs = ct / cluster_sizes
    # Avoid log2(0)
    log_probs = np.where(probs > 0, np.log2(probs), 0.0)
    cluster_entropies = -np.sum(probs * log_probs, axis=1)
    entropy = np.sum((cluster_sizes.flatten() / n_total) * cluster_entropies)

    print(f"=== {model_name} ===")
    print(f"Cluster distribution: {pd.Series(labels).value_counts().sort_index().to_dict()}")
    print(f"Internal Index - Silhouette Score: {sil:.4f}")
    print(f"External Index - Purity:           {purity:.4f}")
    print(f"External Index - Entropy:          {entropy:.4f}\n")

In [ ]:
iris = sns.load_dataset("iris")
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

X_raw = iris[feature_cols].values
y = iris["species"].values

# Standardize features so all 4 measurements contribute equally to Euclidean distance
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f"Full Dataset Shape: {X.shape}")
print("Ground-Truth Class Counts:", pd.Series(y).value_counts().to_dict())

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
kmeans_labels = kmeans.fit_predict(X)

evaluate_clustering(X, kmeans_labels, y, "K-Means (K=3, n_init=10)")

In [ ]:
# eps=0.6 and min_samples=5 on standardized Iris features separates dense core regions
dbscan = DBSCAN(eps=0.6, min_samples=5)
dbscan_labels = dbscan.fit_predict(X)

# Report including noise (-1) as a group, and also excluding noise (-1)
evaluate_clustering(X, dbscan_labels, y, "DBSCAN (eps=0.6, min_samples=5) [All Points]", ignore_noise=False)
evaluate_clustering(X, dbscan_labels, y, "DBSCAN (eps=0.6, min_samples=5) [Core/Border Only]", ignore_noise=True)